# 04 - Block-wise width in AI

Quantized activation with outlier channels. **Layout decides the gain**: per channel (outlier localized in one block) recovers ~2x vs the safe uniform-u16 width; per token (outlier spread across all blocks) stays ~1x. Honest yardstick: byte-granular does not beat flat int8.

In [ ]:
import os, subprocess, tempfile
import matplotlib.pyplot as plt

def find_include():
    d = os.getcwd()
    for _ in range(6):
        cand = os.path.join(d, "include", "smart2raw.h")
        if os.path.exists(cand):
            return os.path.join(d, "include")
        d = os.path.dirname(d)
    raise FileNotFoundError("include/smart2raw.h not found; run from the repo")

INC = find_include()

def compile_run(src, args_list, flags="-O3 -march=native"):
    """Compiles a C harness (with the header) and runs it for each args; returns outputs."""
    with tempfile.NamedTemporaryFile("w", suffix=".c", delete=False) as f:
        f.write(src); cpath = f.name
    exe = cpath[:-2]
    subprocess.check_call(["gcc"] + flags.split() + ["-I", INC, "-o", exe, cpath])
    outs = []
    for a in args_list:
        outs.append(subprocess.check_output([exe] + [str(x) for x in a]).decode().strip())
    os.remove(cpath); os.remove(exe)
    return outs

print("include:", INC)

## Measurement

In [ ]:
SRC = r"""#include <stdio.h>
#include <stdlib.h>
#include "smart2raw.h"
int main(int argc,char**argv){
  double pct=argc>1?atof(argv[1]):1.0; srand(7);
  size_t T=2048,C=512,N=T*C; uint64_t*cm=malloc(N*8),*rm=malloc(N*8);
  size_t nout=0;
  for(size_t c=0;c<C;c++){int o=((double)rand()/RAND_MAX*100.0)<pct; if(o)nout++;
    for(size_t t=0;t<T;t++){uint64_t v=o?(uint64_t)(2000+rand()%25000):(uint64_t)(rand()%201);cm[c*T+t]=v;rm[t*C+c]=v;}}
  S2RBlocked bc,br; s2r_blocked_build(&bc,cm,N,T); s2r_blocked_build(&br,rm,N,C);
  size_t unif=2*N;
  printf("%.3f %zu %zu %zu\n",100.0*nout/C,unif,s2r_blocked_bytes(&bc),s2r_blocked_bytes(&br));
  s2r_blocked_free(&bc);s2r_blocked_free(&br);free(cm);free(rm);return 0;}"""

pcts = [0,0.5,1.0,1.5,2.0,3.0]
rows = [list(map(float, o.split())) for o in compile_run(SRC, [[p] for p in pcts])]
meas   = [r[0] for r in rows]            # measured % of outlier channels
r_chan = [r[1]/r[2] for r in rows]       # uniform u16 / per channel (localized outlier)
r_tok  = [r[1]/r[3] for r in rows]       # uniform u16 / per token (spread outlier)
for m,a,b in zip(meas,r_chan,r_tok): print(f"{m:4.1f}% outliers  per-channel={a:4.2f}x  per-token={b:4.2f}x")

## Chart

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
ax.plot(meas, r_chan, "o-", color="#2E7D5B", lw=2, label="per channel (localized outlier)")
ax.plot(meas, r_tok,  "s--", color="#C2772E", lw=2, label="per token (spread outlier)")
ax.axhline(1.0, color="#888", ls=":")
ax.set_xlabel("% outlier channels"); ax.set_ylabel("memory: uniform u16 / per block")
ax.set_title("AI: block-wise width; layout decides the gain (measured)")
ax.legend(); ax.grid(alpha=0.3); plt.show()

The separation between the two curves is the key point: to gain in AI, the outliers must be localized in memory (per-channel/per-token layout).